# 01 - Statistical Re-analysis of Section IV
**Motivation for the method; prerequisite for everything else.**

The submitted analysis argued that the *shape* of the residual distribution varies spatially. Three weaknesses (found in the audit):

* The ANOVA grouped patches by **image**, so it tested between-image differences, not spatial variation inside an image; patches within an image are not independent, so p < 1e-300 is meaningless.
* The structural features (variance, edge density, entropy) were computed on the **clean image**.
* An obvious alternative was never tested: with **multiplicative speckle the noise scale depends on intensity**. Pooling pixels of different scales gives heavy tails globally, and a patch straddling an edge mixes scales so its fitted beta is lower. This reproduces *all* the reported correlations with **no shape heterogeneity at all**.

This notebook tests that alternative directly, with image-clustered inference.
**Decision rule (written before looking at results):** the heterogeneity-of-shape claim is supported only if the observed patch-beta spread and beta-feature correlations *exceed* what the heteroscedastic-Gaussian null produces.

In [ ]:
import os, sys, json, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import config as C
C.banner()
from scipy import stats
from utils.data import load_npy, downsample
from utils import distfit as D, figures as FG
splits, man = C.get_splits(); tr = splits["train"]     # analysis uses the TRAIN split only (no val/test leakage)
LR = load_npy(tr[0][1]).shape[0]; P = 32 if LR >= 128 else 16
n_img = {"smoke": len(tr), "pilot": 400, "full": len(tr)}[C.MODE]
print("patch size", P, "| images used:", min(n_img, len(tr)))

MODE=pilot  epochs=30  seeds=[0, 1]  device=cpu  out=d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_pilot
patch size 32 | images used: 400


## 1. Global model comparison (replaces Table I)
Homoscedastic families as in the paper, **plus intensity-dependent-scale models**. If a heteroscedastic model beats the global GGD, the 'heavy-tailed' premise must be reframed.

In [ ]:
cmp_, ab = D.global_model_comparison(tr, n_images=min(200, len(tr)))
print(cmp_.round(1).to_string(index=False)); print(ab)
cmp_.to_csv(os.path.join(C.TABLES, "tableI_global_model_comparison.csv"), index=False)
best = cmp_.iloc[0]["model"]
print("\nBest model by AIC:", best, "->", "heteroscedastic structure matters" if best.startswith("Hetero") else "global GGD preferred")

                       model  k    loglik        AIC  dAIC_vs_best
      Hetero-GGD (beta=1.53)  3 4562984.7 -9125963.4           0.0
Hetero-Gaussian  N(0,aI^2+b)  2 4528255.4 -9056506.7       69456.7
                GGD (global)  2 3675453.8 -7350903.7     1775059.7
                     Laplace  1 3651217.5 -7302433.0     1823530.4
                   Student-t  2 3580603.7 -7161203.4     1964760.0
                    Gaussian  1 3284801.5 -6569600.9     2556362.5
{'a': 0.03406723900807617, 'b': 5.1708286740514505e-05, 'ggd_beta': 0.7978815716514408, 'ggd_alpha': 0.04100733841282936, 'n': 3276800}

Best model by AIC: Hetero-GGD (beta=1.53) -> heteroscedastic structure matters


## 2. Patch-level table
Fast moment estimator of beta (checked against the MLE below), residual statistics and *clean-image* features per 32x32 patch.

In [ ]:
tab = D.patch_table(tr, P, max_images=n_img)
print(len(tab), "patches |", f"beta mean={tab.beta.mean():.3f} std={tab.beta.std():.3f}")
chk = []
for g, n in tr[:min(20, len(tr))]:
    gt, ny = load_npy(g), load_npy(n); r = ny - downsample(gt, ny.shape[:2])
    for i in range(0, ny.shape[0] - P + 1, P):
        for j in range(0, ny.shape[1] - P + 1, P):
            p = r[i:i+P, j:j+P].ravel(); chk.append((D.ggd_beta_moment(p[None])[0], stats.gennorm.fit(p, floc=0)[0]))
chk = np.array(chk)
print(f"moment vs MLE beta over {len(chk)} patches: Spearman rho = {stats.spearmanr(chk[:,0], chk[:,1])[0]:.3f}, mean moment={chk[:,0].mean():.3f}, mean MLE={chk[:,1].mean():.3f}")

6400 patches | beta mean=1.450 std=0.445
moment vs MLE beta over 320 patches: Spearman rho = 0.996, mean moment=1.579, mean MLE=1.569


## 3. What does the heterogeneity look like when tested properly?

In [ ]:
vd = D.variance_decomposition(tab)
print("variance of patch beta  BETWEEN images:", round(vd["var_between"], 4), "| WITHIN images:", round(vd["var_within"], 4), "| ICC =", round(vd["ICC"], 3))
feats = ["edge_density", "gt_var", "entropy"]
obs = {f: D.cluster_bootstrap_corr(tab, f, n_boot=300 if C.MODE != "full" else 1000) for f in feats}
part = {f: D.partial_corr(tab, f, "beta", "gt_mean") for f in feats}
for f in feats:
    print(f"r(beta, {f:13s}) = {obs[f]['r']:+.3f}  image-clustered 95% CI {tuple(round(v,3) for v in obs[f]['ci95'])}   |  partial r | local intensity = {part[f]:+.3f}")

variance of patch beta  BETWEEN images: 0.1022 | WITHIN images: 0.096 | ICC = 0.516
r(beta, edge_density ) = -0.373  image-clustered 95% CI (-0.431, -0.319)   |  partial r | local intensity = -0.470
r(beta, gt_var       ) = -0.503  image-clustered 95% CI (-0.539, -0.472)   |  partial r | local intensity = -0.678
r(beta, entropy      ) = -0.420  image-clustered 95% CI (-0.467, -0.374)   |  partial r | local intensity = -0.634


## 4. Null models
**Homogeneous-shape null**: every pixel from the *same* GGD (global beta) - what pure estimation noise gives.  
**Heteroscedastic-Gaussian null**: `r ~ N(0, a I^2 + b)` given the clean intensity - every pixel exactly Gaussian, *no shape heterogeneity*.

In [ ]:
n_null = min(n_img, 300)
null_h = D.homogeneous_null(tr, ab["ggd_beta"], patch=P, max_images=n_null)
null_g = D.heteroscedastic_null(tr, ab["a"], ab["b"], patch=P, max_images=n_null)
table = D.compare_observed_vs_null(tab, {"homogeneous GGD null": null_h, "hetero-Gaussian null": null_g}, feats)
print(table.round(3).to_string(index=False)); table.to_csv(os.path.join(C.TABLES, "section4_observed_vs_null.csv"), index=False)

               table  beta_mean  beta_std  r(beta,edge_density)  r(beta,gt_var)  r(beta,entropy)
            observed      1.450     0.445                -0.373          -0.503           -0.420
homogeneous GGD null      0.805     0.053                -0.044          -0.030           -0.028
hetero-Gaussian null      1.441     0.452                -0.480          -0.562           -0.496


In [ ]:
# ---- decision rule ----
rows = []
for f in feats:
    r_obs, (lo, hi) = obs[f]["r"], obs[f]["ci95"]
    r_null = table.loc[table.table == "hetero-Gaussian null", f"r(beta,{f})"].item()
    rows.append(dict(feature=f, r_observed=r_obs, ci_lo=lo, ci_hi=hi, r_hetero_null=r_null, null_inside_CI=bool(lo <= r_null <= hi)))
dec = pd.DataFrame(rows)
sd_obs, sd_null = tab.beta.std(), null_g.beta.std()
excess_sd = float(np.sqrt(max(sd_obs ** 2 - sd_null ** 2, 0)))
mean_gap = float(tab.beta.mean() - null_g.beta.mean())
print(dec.round(3).to_string(index=False))
print(f"\npatch-beta SD: observed {sd_obs:.3f} vs hetero-Gaussian null {sd_null:.3f}  -> EXCESS SD attributable to genuine shape variation = {excess_sd:.3f}")
print(f"mean patch beta: observed {tab.beta.mean():.3f} vs null {null_g.beta.mean():.3f}  (gap {mean_gap:+.3f})")
dec["observed_exceeds_null"] = dec.ci_hi < dec.r_hetero_null          # observed MORE negative than the null (CI entirely beyond it)
explained = (not dec.observed_exceeds_null.any()) and excess_sd < 0.5 * sd_null
verdict = ("NOT SUPPORTED: signal-dependent (heteroscedastic) Gaussian noise reproduces the heavy-tailed pooled fit, the patch-beta spread and the "
           "feature correlations. Reframe Section IV as 'signal-dependent noise scale', and test the method against a scale-only model."
           if explained else
           "PARTLY/FULLY SUPPORTED: observed heterogeneity exceeds the heteroscedastic-Gaussian null on at least one criterion; "
           "report the excess and keep the shape-heterogeneity motivation, scoped to the quantified excess.")
print("\nVERDICT:", verdict)
json.dump({"global_best_model": best, "ab": ab, "ICC": vd, "observed": obs, "partial_corr_vs_local_intensity": part,
           "excess_sd": excess_sd, "mean_gap": mean_gap, "heterogeneity_of_shape_supported": (not explained), "verdict": verdict},
          open(os.path.join(C.TABLES, "section4_summary.json"), "w"), indent=1, default=float)

     feature  r_observed  ci_lo  ci_hi  r_hetero_null  null_inside_CI
edge_density      -0.373 -0.431 -0.319         -0.480           False
      gt_var      -0.503 -0.539 -0.472         -0.562           False
     entropy      -0.420 -0.467 -0.374         -0.496           False

patch-beta SD: observed 0.445 vs hetero-Gaussian null 0.452  -> EXCESS SD attributable to genuine shape variation = 0.000
mean patch beta: observed 1.450 vs null 1.441  (gap +0.009)

VERDICT: NOT SUPPORTED: signal-dependent (heteroscedastic) Gaussian noise reproduces the heavy-tailed pooled fit, the patch-beta spread and the feature correlations. Reframe Section IV as 'signal-dependent noise scale', and test the method against a scale-only model.


## 5. Figures (caption numbers come from the saved JSON - do not retype them)

In [ ]:
prov = {"split": "train", "n_patches": int(len(tab)), "patch": P, "estimator": "moment (checked vs MLE)",
        "r_observed": {f: obs[f]["r"] for f in feats}}
FG.scatter_beta(tab, feats, C.FIGS, "fig_beta_vs_structure_observed", prov)
FG.scatter_beta(null_g, feats, C.FIGS, "fig_beta_vs_structure_hetero_null", {**prov, "table": "heteroscedastic-Gaussian null"})
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.hist(tab.beta, bins=50, alpha=.6, density=True, label="observed"); ax.hist(null_g.beta, bins=50, alpha=.6, density=True, label="hetero-Gaussian null")
ax.hist(null_h.beta, bins=50, alpha=.6, density=True, label="homogeneous GGD null"); ax.set_xlabel("patch beta"); ax.legend(); ax.set_title("Is patch-beta spread beyond nulls?")
FG.save(fig, C.FIGS, "fig_patch_beta_vs_nulls", {"sd_obs": sd_obs, "sd_null_hetero": sd_null, "excess_sd": excess_sd})
print("figures saved to", C.FIGS)

figures saved to d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_pilot\figures
